# Submission demo

End-to-end dry run of `src.submit.build_submission`:

1. Build a **dummy** PEFT/LoRA adapter directory on disk (config + safetensors weights).
2. Validate it using the packager.
3. Zip it into `submission.zip` (flat layout, as required).
4. Inspect the zip contents, size, and metadata.
5. Verify that over-rank adapters are **rejected** before upload.

**Goal:** catch packaging bugs now, before we have real weights. When Phase 1 SFT finishes, we swap the dummy dir for the trained adapter and the rest of the flow is identical.

In [ ]:
import json
import os
import sys
import zipfile
from pathlib import Path

# Make the repo root importable when the notebook is run from `notebooks/`.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.submit.build_submission import (
    MAX_LORA_RANK,
    ValidationReport,
    build,
    validate,
)

print("repo root:", REPO_ROOT)
print("MAX_LORA_RANK:", MAX_LORA_RANK)

## 1. Build a dummy adapter directory

We emit the same files a real PEFT `save_pretrained(...)` call would produce:

- `adapter_config.json` (PEFT schema, `r=32`, alpha=64, targets q/k/v/o).
- `adapter_model.safetensors` — real safetensors container holding a handful of zero tensors with shapes matching what PEFT would write (`lora_A.weight` = `[r, in]`, `lora_B.weight` = `[out, r]`).
- `README.md` as optional metadata.

The packager only *reads* the config and existence of weights; it never loads the tensors. A dummy tensor with the right shape is enough to exercise the pipeline.

In [ ]:
try:
    from safetensors.torch import save_file
    import torch
    HAS_SAFETENSORS = True
except ImportError:
    HAS_SAFETENSORS = False
    print("safetensors/torch not installed — will fall back to writing a stub .bin file.")

In [ ]:
DEMO_DIR = REPO_ROOT / "outputs" / "demo_adapter"
DEMO_DIR.mkdir(parents=True, exist_ok=True)

# --- adapter_config.json ------------------------------------------------
# This is what PEFT would emit for a QLoRA run on Nemotron-3-Nano-30B.
# Nemotron-3-Nano-30B-A3B is a MoE; we target the standard attention projections.
adapter_config = {
    "auto_mapping": None,
    "base_model_name_or_path": "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
    "bias": "none",
    "fan_in_fan_out": False,
    "inference_mode": True,
    "init_lora_weights": True,
    "layers_pattern": None,
    "layers_to_transform": None,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "modules_to_save": None,
    "peft_type": "LORA",
    "r": 32,
    "revision": None,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "task_type": "CAUSAL_LM",
}

(DEMO_DIR / "adapter_config.json").write_text(
    json.dumps(adapter_config, indent=2), encoding="utf-8"
)

# --- adapter_model.safetensors -----------------------------------------
# Shapes roughly mirror a single transformer layer's LoRA weights
# (hidden size 4096, rank 32). A handful of tensors is enough to exercise
# the packager; real runs will have many more layers.
weights_path = DEMO_DIR / "adapter_model.safetensors"
if HAS_SAFETENSORS:
    R, H = 32, 4096
    stub_tensors = {}
    for proj in adapter_config["target_modules"]:
        stub_tensors[f"base_model.model.model.layers.0.self_attn.{proj}.lora_A.weight"] = \
            torch.zeros(R, H, dtype=torch.bfloat16)
        stub_tensors[f"base_model.model.model.layers.0.self_attn.{proj}.lora_B.weight"] = \
            torch.zeros(H, R, dtype=torch.bfloat16)
    save_file(stub_tensors, str(weights_path))
else:
    # Fall back to adapter_model.bin with a stub payload. The packager
    # accepts either filename; content is never parsed at package time.
    weights_path = DEMO_DIR / "adapter_model.bin"
    weights_path.write_bytes(b"\x00" * 1024)

# --- README.md (optional) ----------------------------------------------
(DEMO_DIR / "README.md").write_text(
    "# Demo adapter\n\n"
    "Dummy PEFT/LoRA adapter used to validate the submission packager.\n"
    "Do NOT submit this to the leaderboard — the weights are all zero.\n",
    encoding="utf-8",
)

print("dummy adapter files:")
for p in sorted(DEMO_DIR.iterdir()):
    print(f"  {p.name:<32s}  {p.stat().st_size:>10,d} bytes")

## 2. Validate

In [ ]:
report = validate(DEMO_DIR)
print(report.summary())

## 3. Build `submission.zip`

In [ ]:
SUBMISSION_PATH = REPO_ROOT / "outputs" / "submission.zip"
report = build(DEMO_DIR, SUBMISSION_PATH)
print(f"wrote {SUBMISSION_PATH} ({SUBMISSION_PATH.stat().st_size / 1e6:.2f} MB)")

## 4. Inspect the zip

The zip must use a **flat layout** — all files live at the root of the archive, not nested inside a directory. This matches the NVIDIA reference submission.

In [ ]:
with zipfile.ZipFile(SUBMISSION_PATH) as zf:
    infos = zf.infolist()
    print(f"{'name':<36s} {'size':>10s} {'compressed':>12s}")
    print("-" * 62)
    for info in infos:
        print(f"{info.filename:<36s} {info.file_size:>10,d} {info.compress_size:>12,d}")
    # Sanity: no nested paths
    assert all("/" not in i.filename and "\\" not in i.filename for i in infos), \
        "zip layout is not flat"
    # Required files are present
    names = {i.filename for i in infos}
    assert "adapter_config.json" in names
    assert ("adapter_model.safetensors" in names) or ("adapter_model.bin" in names)
    print("\nOK — zip layout is flat and contains required files.")

## 5. Negative test: rank > 32 must be rejected

Catch the most common footgun: an over-rank adapter would be silently accepted by Kaggle and then fail at vLLM load time (`max_lora_rank=32`). Our packager must raise **before** the zip is written.

In [ ]:
BAD_DIR = REPO_ROOT / "outputs" / "demo_adapter_bad"
BAD_DIR.mkdir(parents=True, exist_ok=True)
bad_cfg = dict(adapter_config)
bad_cfg["r"] = 64  # over the limit
(BAD_DIR / "adapter_config.json").write_text(json.dumps(bad_cfg, indent=2), encoding="utf-8")
(BAD_DIR / "adapter_model.safetensors").write_bytes(b"\x00" * 128)

try:
    validate(BAD_DIR)
except ValueError as e:
    print(f"correctly rejected: {e}")
else:
    raise AssertionError("validate() should have rejected r=64")

## Summary

- `outputs/demo_adapter/` — dummy adapter directory (safe to delete).
- `outputs/submission.zip` — **the exact format we will upload** once Phase 1 finishes.
- The packager validates rank ≤ 32, required files, and flat layout before writing.

When real weights land, the flow is:

```
python -m src.submit.build_submission \
    --adapter-dir outputs/phase1_qlora/adapter \
    --output submission.zip
```